# 03 · Clustering
Standardisasi → penentuan k → K-Means (utama) + Ward (pembanding) → ARI.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *
from src.preprocessing import scale_features
from src.clustering import evaluate_k, fit_kmeans, fit_ward, agreement
from src.visualization import save_matplotlib

df = pd.read_csv(MASTER_FILE).dropna(subset=FEATURES).reset_index(drop=True)
X, scaler = scale_features(df, FEATURES)

## Penentuan k: Elbow, Silhouette, Davies-Bouldin

In [ ]:
scores = evaluate_k(X)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["inertia", "silhouette", "davies_bouldin"]):
    ax.plot(scores["k"], scores[col], marker="o")
    ax.set_title(col)
    ax.set_xlabel("k")
save_matplotlib(fig, "03_penentuan_k")
scores

Pilih k yang seimbang secara statistik **dan** mudah ditafsirkan. Alasan: _tulis di sini_

In [ ]:
K = 4
km = fit_kmeans(X, K)
Z, ward_labels = fit_ward(X, K)
df["cluster"] = km.labels_
df["cluster_ward"] = ward_labels
print("ARI K-Means vs Ward:", round(agreement(km.labels_, ward_labels), 3))

## Dendrogram

In [ ]:
from scipy.cluster.hierarchy import dendrogram
fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z, labels=df["provinsi"].tolist(), leaf_rotation=90, ax=ax)
save_matplotlib(fig, "03_dendrogram")

## PCA 2D

In [ ]:
from sklearn.decomposition import PCA
import plotly.express as px
pc = PCA(n_components=2, random_state=RANDOM_STATE).fit(X)
coords = pc.transform(X)
print("Explained variance:", pc.explained_variance_ratio_.round(3))
px.scatter(x=coords[:, 0], y=coords[:, 1], color=df["cluster"].astype(str), text=df["provinsi"],
           labels={"x": "PC1", "y": "PC2", "color": "cluster"})

## (Opsional) K-Medoids untuk cek sensitivitas outlier

In [ ]:
# from sklearn_extra.cluster import KMedoids
# kmed = KMedoids(n_clusters=K, random_state=RANDOM_STATE).fit(X)
# print("ARI K-Means vs K-Medoids:", agreement(km.labels_, kmed.labels_))

In [ ]:
df.to_csv(CLUSTER_FILE, index=False)